# 🧩 Prerequisites

In [ ]:
# @title 📀 Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
%cd /content/drive/MyDrive/colab/thesis/hqs
CWD = Path.cwd()

In [ ]:
# @title 📦 Package Installation
%%capture
!pip install uv
!uv pip install evaluate rouge_score bert-score
!uv pip install ctranslate2
!uv pip install transformers==4.56.2

# 📊 Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("adasatwork-thesis/hqs", data_files={
    "test" : "test.jsonl"
}, split="test")

print(dataset)

# 🧬 Pegasus Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from typing import List, Union

class Pegasus:
    def __init__(self, model_name: str, batch_size: int = 16):
        self.device = torch.device("cuda")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")
        self.model.eval()
        self.batch_size = batch_size

    def generate(self, texts: Union[str, List[str]]) -> Union[str, List[str]]:
        # Allow passing either a single string or a list of strings
        is_single_string = isinstance(texts, str)
        if is_single_string:
            texts = [texts]

        results = self.generate_in_batches(texts)
        return results[0] if is_single_string else results

    def generate_in_batches(self, texts: List[str]) -> List[str]:
        all_outputs = []

        # Process inputs in chunks
        for i in range(0, len(texts), self.batch_size):
            batch_texts = texts[i : i + self.batch_size]

            inputs = self.tokenizer(
                batch_texts,
                max_length=512,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(self.device)

            with torch.inference_mode():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    outputs = self.model.generate(
                        **inputs,
                        max_length=256,
                        num_beams=4,
                        length_penalty=0.8,
                        repetition_penalty=1.5,
                    )

            # Use batch_decode instead of looping over outputs
            decoded_batch = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_outputs.extend(decoded_batch)

        return all_outputs

# 🔮 Qwen3 Model

In [ ]:
# @title Qwen3 Inference Using CTranslate2
import re
import ctranslate2
from transformers import AutoTokenizer

clean_pattern = re.compile(r"(?:\n|/\\|\\/|\s+)")

system_text = """You are a Medical Hallucination Detector and Corrector."""
user_text = """\
# Instructions:
1. Compare the [ORIGINAL QUESTION] with its [QUESTION SUMMARY].
2. Identify the incorrect detail in the [QUESTION SUMMARY] and fix it using the [ORIGINAL QUESTION].
3. Output ONLY the final [CORRECTED QUESTION SUMMARY] in fewer than 15 words.

# Context:
[ORIGINAL QUESTION]: {}
[QUESTION SUMMARY]: {}

# Output:
[CORRECTED QUESTION SUMMARY]:"""


class Qwen3:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.generator = ctranslate2.Generator(model_name, device="cuda")

    def generate(self, sources, pegasus_gens):
        # Build the static (system) prompt tokens
        system_messages = [{"role" : "system", "content" : system_text}]
        system_prompt = self.tokenizer.apply_chat_template(
            system_messages, tokenize=False, add_generation_prompt=False
        )
        system_tokens = self.tokenizer.convert_ids_to_tokens(
            self.tokenizer.encode(system_prompt, add_special_tokens=False)
        )

        # Build per-request user prompt tokens
        batch_tokens = []
        for source, pegasus_gens in zip(sources, pegasus_gens):
            messages = [{
                "role" : "user",
                "content" : user_text.format(
                    clean_pattern.sub(" ", source).strip(), pegasus_gens)
            }]

            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            input_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
            batch_tokens.append(self.tokenizer.convert_ids_to_tokens(input_ids))

        # Run batch generation
        results = self.generator.generate_batch(
            batch_tokens,
            static_prompt=system_tokens,
            max_length=64,
            max_batch_size=1,
            sampling_topk=1, # Greedy decoding for the fastest possible output
            include_prompt_in_result=False
        )

        # Decode and output
        outputs = self.tokenizer.batch_decode(
            [r.sequences_ids[0] for r in results],
            skip_special_tokens=True
        )

        return outputs

# 🔴 Final Model

In [ ]:
pega = Pegasus("models/finetuned_pegasus")
qwen = Qwen3("models/qwen3-sft-merged-ct2")
qwnx = Qwen3("models/qwen3-simpo-merged-ct2")

In [ ]:
import pandas as pd
test_df = dataset.to_pandas()

In [ ]:
sources = test_df["source"].to_list()
peggens = pega.generate(sources)

In [ ]:
qwngens = qwen.generate(sources, peggens)

In [ ]:
qwngenx = qwnx.generate(sources, peggens)

In [ ]:
test_df["pegasus_gen"] = peggens
test_df["qwen3_gen"] = qwngens
test_df["qwen3x_gen"] = qwngenx
test_df.info()

In [ ]:
test_df.to_json("data/test-gen3.jsonl", orient="records", lines=True)